In [8]:
#Mohammad_Abdulrouf_Alnaghmoush
#2230004028
#9M01
# Install the missing emoji package directly inside the notebook
!pip install emoji nltk pandas

import re
import emoji
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
from nltk.lm.preprocessing import padded_everygram_pipeline
from nltk.lm import MLE

# Download required tokenizers from NLTK
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\aanna\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\aanna\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [9]:
# Define dataset path
file_path = 'Random Tweets from Pakistan- Cleaned- Anonymous.csv'

# Load CSV dataset using UTF-8 encoding
df = pd.read_csv(file_path, encoding='utf-8', encoding_errors='replace')

# Verify total loaded rows and display initial records
print(f"Total rows loaded: {df.shape[0]}")
df[['full_text']].head()

Total rows loaded: 202202


C:\Users\aanna\AppData\Local\Temp\ipykernel_6028\829439583.py:5: DtypeWarning: Columns (0: reply_count) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, encoding='utf-8', encoding_errors='replace')


,full_text
0,تیرا لیڈر میرا لیڈر نواز شریف نواز شریف❤️ کیا...
1,"Happy birthday to my brother n boss , May you ..."
2,❤️❤️
3,`suspicious °jikook au jimin'in yaşadığı kasab...
4,Speaking of @SpiderMan... 😂 https://t.co/uGJ...


In [10]:
def clean_tweet(text):
    """
    Cleans raw tweet text according to the lab requirements:
    Removes RT, URLs, mentions (@), hashtags (#), and emojis.
    """
    if not isinstance(text, str):
        return ""
    
    # 1. Remove retweet indicators
    text = re.sub(r'\bRT\b', '', text)
    
    # 2. Remove web links / URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    
    # 3. Remove user mentions
    text = re.sub(r'@\w+', '', text)
    
    # 4. Remove hashtags
    text = re.sub(r'#\w+', '', text)
    
    # 5. Remove emojis using the installed emoji package
    text = emoji.replace_emoji(text, replace='')
    
    # Normalize excess whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply cleaning to the full_text column
cleaned_series = df['full_text'].dropna().apply(clean_tweet)

# Filter out empty records
cleaned_tweets = [t for t in cleaned_series if len(t) > 0]

# Limit the sample (e.g., 20,000 tweets) to prevent memory overload during training
sample_size = min(20000, len(cleaned_tweets))
tokenized_text = [word_tokenize(tweet.lower()) for tweet in cleaned_tweets[:sample_size]]

print(f"Total cleaned & tokenized tweets ready for training: {len(tokenized_text)}")

Total cleaned & tokenized tweets ready for training: 20000


In [11]:
# Set n-gram order to 2 for Bigram modeling
n = 2

# Create the padded pipeline (n-grams generator and padded vocabulary)
train_data, padded_vocab = padded_everygram_pipeline(n, tokenized_text)

# Initialize Maximum Likelihood Estimation (MLE) model
lm = MLE(n)

# Fit model on the training data
lm.fit(train_data, padded_vocab)

print(f"Bigram MLE Model successfully trained.")
print(f"Total vocabulary size: {len(lm.vocab)}")

Bigram MLE Model successfully trained.
Total vocabulary size: 42572


In [12]:
# Generate a sequence of tokens from the trained language model
generated_tokens = lm.generate(num_words=15, random_seed=42)

# Remove padding boundary tags from the output string
generated_tweet = ' '.join([w for w in generated_tokens if w not in ['<s>', '</s>']])
print("Generated Tweet Sample:\n", generated_tweet)

# Evaluate model perplexity on sample bigrams
test_sample = [('in', 'pakistan'), ('the', 'government')]
test_perplexity = lm.perplexity(test_sample)
print(f"\nPerplexity on test sequence {test_sample}: {test_perplexity:.4f}")

Generated Tweet Sample:
 that 's favorite cricketer he played till by its a customer gifted batsmen for some

Perplexity on test sequence [('in', 'pakistan'), ('the', 'government')]: 50.5425


In [13]:
# Step 5: Calculate the conditional probability P('is' | 'pakistan')
prob_is_given_pakistan = lm.score('is', ['pakistan'])
print(f"P('is' | 'pakistan') = {prob_is_given_pakistan:.6f}")

# Step 6: Calculate the perplexity of the word 'pakistan'
perplexity_pakistan = lm.perplexity([('pakistan',)])
print(f"Perplexity of ('pakistan'): {perplexity_pakistan:.4f}")

P('is' | 'pakistan') = 0.032165
Perplexity of ('pakistan'): 217.2636
